# KV-cache pruning experiments on Kaggle (2x T4)

**Before running:** Settings > Accelerator = **GPU T4 x2**, Internet = **On**. Phone-verify your Kaggle account to unlock GPUs.

Kaggle sessions stop after 12 hours. Every run **appends** to `results/*.jsonl` and **skips rows already done**, so to continue in a new session: download `results.zip` at the end, upload it as a Kaggle Dataset, attach it, and run the *Restore* cell before anything else.

In [ ]:
!nvidia-smi

In [ ]:
REPO = "https://github.com/khushpatel3016/kv-cache-entropy-pruning.git"   # <-- change
!git clone -q $REPO /kaggle/working/kvp
%cd /kaggle/working/kvp
!pip install -q "transformers==5.20.1" "datasets==3.6.0" rapidfuzz tabulate pyyaml
!pip install -q -e .


### Restore earlier results (only when continuing a previous session)

In [ ]:
import glob, shutil, os
for f in glob.glob("/kaggle/input/*/results/*.jsonl") + glob.glob("/kaggle/input/*/*.jsonl"):
    shutil.copy(f, "results/"); print("restored", f)
for f in glob.glob("/kaggle/input/*/profiles/*.json"):
    shutil.copy(f, "profiles/"); print("restored", f)

### Hugging Face login (only needed for Llama-3.2)
Add your token under Add-ons > Secrets as `HF_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    print("no HF token (fine for Qwen):", e)

### 1. Tests (about 10 seconds)

In [ ]:
!python -m pytest -q

### 2. Static entropy profile (held-out LongBench tasks, about 10 minutes)

In [ ]:
import os
if not os.path.exists("profiles/qwen2.5-3b-instruct.json"):
    !python scripts/calibrate_profile.py --model Qwen/Qwen2.5-3B-Instruct --out profiles/qwen2.5-3b-instruct.json --per-task 9 --max-len 8192

### 3. Main experiment, one process per GPU
Run budget 0.2 first. Then change `BUDGET` to 0.1, 0.3, 0.5 and run this cell again.

In [ ]:
import subprocess, time
BUDGET = "0.2"
splits = {0: ["qasper", "hotpotqa", "gov_report"], 1: ["trec", "passage_retrieval_en", "lcc"]}
procs = []
for gpu, tasks in splits.items():
    cmd = ["python", "-m", "kvprune.run", "--config", "configs/main_qwen3b.yaml", "--budgets", BUDGET,
           "--tasks", *tasks, "--output", f"results/main_qwen3b_gpu{gpu}.jsonl"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONPATH": "src"}
    procs.append(subprocess.Popen(cmd, env=env, stdout=open(f"log_gpu{gpu}.txt", "a"), stderr=subprocess.STDOUT))
while any(p.poll() is None for p in procs):
    time.sleep(120)
    for g in splits:
        print(g, open(f"log_gpu{g}.txt").read().strip().splitlines()[-1:] )
print("done", [p.returncode for p in procs])

### 4. Passkey, ablations, profiling (run when the main budgets are done)

In [ ]:
%env PYTHONPATH=src
!CUDA_VISIBLE_DEVICES=0 python -m kvprune.run --config configs/passkey_qwen3b.yaml > log_pk.txt 2>&1 &
!CUDA_VISIBLE_DEVICES=1 python -m kvprune.run --config configs/ablation_qwen3b.yaml > log_ab.txt 2>&1 &

In [ ]:
# After the two jobs above finish:
!python scripts/profile_memory.py --model Qwen/Qwen2.5-3B-Instruct --lengths 4096 8192 16384 --out results/profile_qwen3b.jsonl
!python -m kvprune.run --config configs/ablation_window16_qwen3b.yaml
!python -m kvprune.run --config configs/ablation_window64_qwen3b.yaml

### 5. Figures and download

In [ ]:
!python scripts/make_figures.py --results "results/*.jsonl" --out figures
!zip -qr /kaggle/working/results.zip results profiles figures
print("Download /kaggle/working/results.zip from the Output panel, then commit results/ and figures/ to GitHub.")